# PlayTrain quickstart

PlayTrain is an RL framework for video-game environments that are generated and modified
by a language model. Every environment is a single JavaScript file: a person can play it,
and an agent can train on the same game.

This notebook walks the whole loop:

1. make an environment and step it
2. **read the game's source and edit it** - the part you can't do with ALE or ProcGen
3. measure throughput on this machine
4. train IMPALA on `breakout`
5. watch a policy trained on the full config actually play

**Runtime -> Change runtime type -> GPU** before you start. Step 4 wants one.

### About the numbers you'll see here

Colab gives you ~2 vCPU and one small GPU. The paper's headline figures come from an
80-thread node. Nothing here will reproduce them, and it isn't trying to - step 3
measures *per-core* throughput and a *same-machine* comparison, which are the parts
that transfer. Absolute numbers are noted where they differ and why.

## 0. Install

Builds the native runtime from source (~4-6 min, mostly compiling QuickJS). Run once.

In [ ]:
%%bash
set -euo pipefail

# clang for the runtime, cargo for the rasterizer. Colab has neither reliably.
apt-get -qq install -y clang >/dev/null
command -v cargo >/dev/null || {
  curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable >/dev/null
}
export PATH="$HOME/.cargo/bin:$PATH"

# uv: the project's installer. Installing into Colab's system Python reuses the
# preinstalled torch instead of pulling down a fresh 2 GB copy.
command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh >/dev/null
export PATH="$HOME/.local/bin:$PATH"

# Into /content/src: a directory named `playtrain` next to the kernel's cwd
# shadows the installed package, and the trainers expect it as a sibling.
mkdir -p /content/src && cd /content/src
[ -d playtrain ]          || git clone -q --depth 1 https://github.com/heyodog0/playtrain.git
[ -d playtrain-trainers ] || git clone -q --depth 1 https://github.com/heyodog0/playtrain-trainers.git

# The native backend: rasterizer + QuickJS staticlibs, then the vectorized .so.
# (build_qjs.sh shallow-clones quickjs-ng and openlibm on first run.)
cd /content/src/playtrain
bash native/build_qjs.sh
bash native/build_qjs_vec.sh

uv pip install --system -q -e /content/src/playtrain
uv pip install --system -q -e /content/src/playtrain-trainers
echo "OK"

In [ ]:
import os, sys
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

import numpy as np, torch
from playtrain.runtime import GameEnv, NativeVecEnv, list_available_games

print(f"{len(list_available_games())} games:", ", ".join(sorted(list_available_games())[:12]), "...")
print("cpus:", os.cpu_count(), "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. An environment

Ordinary Gymnasium. `reset`, `step`, a `Box(0,255,(64,64,3))` observation, a
`Discrete(8)` action space. Nothing to learn here - that's the point.

In [ ]:
import matplotlib.pyplot as plt

env = GameEnv(game="breakout", obs_size=64)
print("observation:", env.observation_space)
print("action:     ", env.action_space)

obs, info = env.reset(seed=0)
frames = [obs.copy()]
for _ in range(120):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    frames.append(obs.copy())

fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for ax, i in zip(axes, np.linspace(0, len(frames) - 1, 6).astype(int)):
    ax.imshow(frames[i]); ax.set_title(f"t={i}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. The environment is a file you can read

Every game is a self-contained p5-style JavaScript file, usually a couple hundred lines.
You can read it, diff it, and change it. This is what the catalog is *for* - the games
are meant to be modified, not just consumed.

In [ ]:
from pathlib import Path

src = Path("/content/src/playtrain/examples/games/js/breakout.js").read_text()
print(f"{len(src.splitlines())} lines\n")
print("\n".join(src.splitlines()[:45]))

In [ ]:
# Change one number and you have a new environment. Here: a much wider paddle,
# the `w: 80` in the `paddle = {...}` block printed above.
import re, tempfile

variant_src, n = re.subn(r"^(    w: )80,$", r"\g<1>200,", src, count=1, flags=re.M)
assert n == 1, "line not found - open the source above and pick a number to edit"

variant_path = Path(tempfile.mkdtemp()) / "breakout_widepaddle.js"
variant_path.write_text(variant_src)

def first_frames(game, n=90, seed=0):
    e = GameEnv(game=str(game), obs_size=64)
    o, _ = e.reset(seed=seed)
    for _ in range(n):
        o, *_ = e.step(e.action_space.sample())
    e.close()
    return o

fig, (a, b) = plt.subplots(1, 2, figsize=(6, 3))
a.imshow(first_frames("breakout"));   a.set_title("breakout");        a.axis("off")
b.imshow(first_frames(variant_path)); b.set_title("2.5x paddle width");  b.axis("off")
plt.tight_layout(); plt.show()

## 3. Speed

Two measurements that mean something on a 2-core VM:

- **per-core throughput** - steps/s/core, which is hardware-relative
- **scaling** across the cores you have

Then a same-machine comparison against ALE, since a ratio measured on one box is worth
more than two absolute numbers measured on different ones.

In [ ]:
import time

def measure(num_envs, num_threads, steps=400, game="breakout"):
    venv = NativeVecEnv(game=game, num_envs=num_envs, num_threads=num_threads, obs_size=64)
    venv.reset(0)
    acts = np.zeros(num_envs, dtype=np.int64)
    for _ in range(20):                       # warmup
        venv.step(acts)
    t0 = time.perf_counter()
    for _ in range(steps):
        venv.step(acts)
    dt = time.perf_counter() - t0
    venv.close()
    return steps * num_envs / dt

!lscpu | egrep 'Model name|^CPU\\(s\\)|Thread|Core'

print(f"\\nsingle env, single thread: {measure(1, 1, steps=2000):,.0f} steps/s")


In [ ]:
# Same machine, same wall clock: PlayTrain vs ALE.
!uv pip install --system -q "ale-py>=0.10" autorom >/dev/null 2>&1
!python -c "import ale_py" && AutoROM --accept-license -q >/dev/null 2>&1 || true

import gymnasium as gym

def ale_sps(steps=2000):
    import ale_py; gym.register_envs(ale_py)
    e = gym.make("ALE/Breakout-v5", frameskip=1)
    e.reset(seed=0)
    for _ in range(50): e.step(0)
    t0 = time.perf_counter()
    for _ in range(steps):
        _, _, term, trunc, _ = e.step(e.action_space.sample())
        if term or trunc: e.reset()
    return steps / (time.perf_counter() - t0)

pt = measure(num_envs=1, num_threads=1, steps=2000)
try:
    ale = ale_sps()
    print(f"PlayTrain  {pt:>10,.0f} steps/s   (1 env, 1 thread)")
    print(f"ALE        {ale:>10,.0f} steps/s   (1 env)")
    print(f"ratio      {pt/ale:>10.2f}x")
except Exception as exc:
    print("ALE unavailable on this runtime:", exc)
    print(f"PlayTrain  {pt:,.0f} steps/s (1 env, 1 thread)")

### Reading these numbers

Per-core throughput and the scaling efficiency are what transfer off this VM. The
absolute totals do not: the paper measures 80 threads on a single node, roughly 40x the
parallelism available here, and uses per-game AOT-compiled engine builds that are too
slow to produce inside a notebook. Expect this cell to land well under the published
figures for that reason, not because the measurement is different.

## 4. Train

IMPALA (V-trace) on `breakout`, small enough to finish in a few minutes.

**Watch the throughput and the loss, not the reward.** This VM has one physical core,
and a run of 1.79M steps here moves mean episode return from 87.5 to 95.5 against a
random baseline of ~79 - i.e. nowhere. The checkpoint in step 5 took **100M steps at 60
env threads**, and the same code reaches ~1M FPS on 4xH100. What this cell demonstrates
is that the loop is wired end to end and what it costs per step; step 5 shows the result.


In [ ]:
import json, subprocess, threading

ncpu = os.cpu_count()
cfg = {
    "game": "breakout", "env_backend": "playtrain",
    "total_steps": 300_000,
    "batch_size": 32, "unroll_length": 64, "num_learner_threads": 1,
    "discounting": 0.99, "baseline_cost": 0.5, "entropy_cost": 0.01,
    "reward_clipping": "abs_one", "grad_norm_clipping": 40.0,
    "learning_rate": 0.0005, "seed": 0, "device": "auto",
    "obs_shape": [3, 64, 64], "num_actions": 8, "features_dim": 256,
    "net": "impala", "use_lstm": False, "frame_skip": 1,
    "inference_mode": "vec",
    "vec_workers": max(1, ncpu - 1), "vec_env_threads": 2, "vec_double_buffer": True,
    "stats_log_every": 20, "eval_every_steps": 0, "save_every_steps": 0,
    "resume": "off", "log_dir": "/content/outputs/impala_colab",
}
Path("/content/impala_colab.json").write_text(json.dumps(cfg, indent=1))
print(json.dumps(cfg, indent=1))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/outputs/impala_colab/tb

In [ ]:
# Runs in the foreground so you can watch it; the TensorBoard panel above updates live.
# Stop early with the interrupt button - the run checkpoints as it goes.
!cd /content/playtrain-trainers && python -m playtrain_trainers.train_impala --config /content/impala_colab.json

## 5. Watch a trained policy

This checkpoint comes from the full config: 100M steps, 12 workers x 5 env threads.
It's 2.5 MB.

In [ ]:
CKPT_URL = "https://github.com/heyodog0/playtrain/releases/download/v0.1.0/impala_breakout_100M.pt"
ckpt_path = "/content/impala_breakout_100M.pt"

import urllib.request
if not Path(ckpt_path).exists():
    urllib.request.urlretrieve(CKPT_URL, ckpt_path)

from playtrain_trainers.impala.net import ImpalaNet

state = torch.load(ckpt_path, map_location="cpu", weights_only=False)["model_state_dict"]
net = ImpalaNet(observation_shape=(3, 64, 64), num_actions=8, features_dim=256,
                use_lstm=False, net="impala")
net.load_state_dict(state)
net.eval()

BLANK = {"reward": torch.zeros(1, 1), "done": torch.zeros(1, 1, dtype=torch.bool),
         "last_action": torch.zeros(1, 1, dtype=torch.int64)}

def act(obs):
    frame = torch.as_tensor(np.ascontiguousarray(obs.transpose(2, 0, 1)))[None, None]
    with torch.no_grad():
        out, _ = net({"frame": frame, **BLANK}, ())
    return int(out["policy_logits"][0, 0].argmax())

def episode(policy, seed, keep=False):
    e = GameEnv(game="breakout", obs_size=64)
    obs, _ = e.reset(seed=seed)
    total, frames = 0.0, [obs.copy()]
    for t in range(3000):
        obs, r, term, trunc, _ = e.step(policy(obs))
        total += r
        if keep: frames.append(obs.copy())
        if term or trunc: break
    e.close()
    return total, t + 1, frames

rng = np.random.default_rng(0)
rand_ret = [episode(lambda o: int(rng.integers(8)), s)[:2] for s in range(8)]
pol_ret  = [episode(act, s)[:2] for s in range(8)]

print(f"random   return {np.mean([r for r, _ in rand_ret]):7.1f}   episode length {np.mean([l for _, l in rand_ret]):6.0f}")
print(f"trained  return {np.mean([r for r, _ in pol_ret]):7.1f}   episode length {np.mean([l for _, l in pol_ret]):6.0f}")

In [ ]:
# Render it playing.
from PIL import Image
from IPython.display import Image as ShowImage, display

_, _, frames = episode(act, seed=0, keep=True)
gif = "/content/breakout_trained.gif"
imgs = [Image.fromarray(f).resize((256, 256), Image.NEAREST) for f in frames[::4]]
imgs[0].save(gif, save_all=True, append_images=imgs[1:], duration=40, loop=0)
display(ShowImage(filename=gif))

## Where to go from here

- **The catalog** - `games/js/` holds every environment as editable source, and
  `GAME_TEMPLATE.md` is the contract a new one has to satisfy.
- **Variants** - several games ship deliberate variants (`frostbite.jungle.js`,
  `caveflyer.blockbreak.js`) built by changing the base game's source. That's the
  intended workflow for generalization studies.
- **Scale** - the training path here is the same one that reaches ~1M FPS on 4xH100
  with the full-node config; the only differences are worker counts and hardware.
  See `playtrain-trainers/configs/` for the configs that produce those numbers.
- **Vectorized envs** - `NativeVecEnv` is the throughput path used above;
  `NativeVectorEnv` wraps it in the Gymnasium `VectorEnv` API if you'd rather
  plug into an existing trainer.